# Project 4 — Financial product recommender

Eligibility first, ranking second, explanations last. Every step calls the same `src/` code as `python -m src.train` and the API.

> Learning project on synthetic data. Not financial advice, underwriting or real bank policy.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_data

raw_users, raw_products, raw_interactions = load_data()
print("users:", raw_users.shape, "products:", raw_products.shape, "interactions:", raw_interactions.shape)

## Inspect columns and missing values
The guide's example fields (`region` on products, `min_age`, `segment`) don't exist here. The real constraints are `min_income` and `risk_level`.

In [ ]:
for name, frame in [("USERS", raw_users), ("PRODUCTS", raw_products), ("INTERACTIONS", raw_interactions)]:
    print(name, frame.columns.tolist())
    print(frame.isna().sum().to_dict(), "\n")
raw_products

## Internal schema
`USER_MAP`, `PRODUCT_MAP` and `INTERACTION_MAP` in `src/config.py` rename columns once. `label = 1` means the user **applied**.

In [ ]:
from src.data import load_prepared

users, products, interactions = load_prepared(include_new_interactions=False)
print(interactions["event_type"].value_counts())
interactions.head()

## Which rules does the data support?
No logged event breaks either rule below, while 3.6% of all user/product pairs would fail the income rule. Low-risk users never touch high-risk products.

In [ ]:
events = interactions.merge(users, on="user_id").merge(products, on="product_id")
print("events meeting min_income:", (events.monthly_income >= events.min_income).mean())
pd.crosstab(events.risk_profile, events.risk_level)

## Eligibility layer, tested with no model

In [ ]:
from src.eligibility import RULES, check_eligibility, eligibility_report

for rule in RULES:
    print(rule["rule_id"], "-", rule["description"])

sample_user = users[users.risk_profile == "low"].iloc[0].to_dict()
report = eligibility_report(sample_user, products)
report[["product_id", "name", "risk_level", "min_income", "eligible", "failed_rules"]]

## Does behaviour depend on the profile?
Barely. The apply rate is about 9.4% everywhere. Expect the ranker to land near chance; the rule layer is what matters.

In [ ]:
events["applied"] = (events.event_type == "apply").astype(int)
print(events.groupby("employment")["applied"].mean().round(3))
print(events.groupby("risk_profile")["applied"].mean().round(3))

## Training examples and point-in-time features
Positives are applications. Negatives are other **eligible** products for the same user and moment, so the ranker only learns to order what the rules allow.

In [ ]:
from src.config import NEGATIVES_PER_POSITIVE, RANDOM_SEED
from src.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES, build_feature_frame, build_training_examples

training_requests = build_training_examples(interactions, users, products, NEGATIVES_PER_POSITIVE, RANDOM_SEED)
training_frame = build_feature_frame(training_requests, users, products, interactions)
training_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["label"]].head()

In [ ]:
from src.model import build_ranker, coefficient_table

ranker = build_ranker(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
ranker.fit(training_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES], training_frame["label"])
coefficient_table(ranker).head(10)

## Eligibility → ranking → explanation

In [ ]:
from src.recommender import FinancialRecommender

recommender = FinancialRecommender(ranker, NUMERIC_FEATURES, CATEGORICAL_FEATURES, users, products, interactions)
result = recommender.recommend(sample_user["user_id"], k=5, include_unfiltered=True, now=pd.Timestamp("2026-09-30T12:00Z"))
print("eligible:", result["eligibility"]["eligible_products"], "of", result["eligibility"]["total_products"])
for product in result["recommendations"]:
    print(f"{product['product_id']} {product['name']:<20} {product['score']:.3f}", [reason["text"] for reason in product["reasons"]])

In [ ]:
pd.DataFrame(result["unfiltered_ranking"])  # what the model alone would have shown

## Temporal evaluation: quality and policy metrics

In [ ]:
from src.evaluate import evaluate

evaluation = evaluate(interactions, users, products)
pd.DataFrame(evaluation["strategies"]).T

## Experiments (guide sections 33 and 44)

In [ ]:
from src.experiments import main as run_experiments
run_experiments()

## Save the artifact
Same as `python -m src.train`.

In [ ]:
from src.train import train
_, metadata = train()
{key: metadata[key] for key in ["model_version", "eligibility_rules_version", "num_training_rows"]}